In [ ]:
%pip install -q -U git+https://github.com/huggingface/transformers accelerate pillow qwen-vl-utils "datasets==3.6.0"


In [ ]:
from pathlib import Path
import torch
from PIL import Image
from qwen_vl_utils import process_vision_info
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"
OUTPUT_DIR = Path("qwen_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
QUESTIONS = {
    42: "What color are the gym shoes?",
    73: "Is this a motorcycle or bike?",
    74: "Does this dog have a collar?",
    133: "What color is lamp?",
    136: "Is this in a museum?",
    139: "What is the woman in the room doing?",
    143: "How many birds are in the tree?",
    164: "What is the color of the refridgerator?",
    192: "What sport is being played?",
    196: "What is the yellow food?",
    208: "What material is the countertop made of?",
    241: "What is he sitting on?",
    257: "Is the dog real?",
    283: "What is the brand of wine?",
    285: "Is it daytime?",
    294: "Are these people having fun?",
    328: "What are the men sitting on?",
    338: "Is this a bedroom?",
    357: "Who are the men in black?",
    359: "Overcast or sunny?",
    360: "Are the streetlamps on?",
}

image_paths = sorted(Path(".").glob("*.jpg"))
samples = [(int(p.stem.rsplit("_", 1)[-1]), p, QUESTIONS[int(p.stem.rsplit("_", 1)[-1])]) for p in image_paths]


In [ ]:
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map="cuda", attn_implementation="eager"
)
processor = AutoProcessor.from_pretrained(MODEL_NAME)
print(torch.__version__, torch.version.cuda, torch.cuda.is_available())


In [ ]:
def make_inputs(image_path, question):
    image = Image.open(image_path).convert("RGB")
    messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": question}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt")
    return {k: v.to(model.device) if hasattr(v, "to") else v for k, v in inputs.items()}

def collect_sample(sample_id, image_path, question, max_new_tokens=32):
    inputs = make_inputs(image_path, question)
    with torch.no_grad():
        generated = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        outputs = model(**inputs, output_attentions=True, return_dict=True)
    answer_ids = [out[len(inp):] for inp, out in zip(inputs["input_ids"], generated)]
    answer = processor.batch_decode(answer_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
    bundle = {
        "sample_id": sample_id,
        "question": question,
        "model_answer": answer,
        "input_ids": inputs["input_ids"][0].detach().cpu(),
        "tokens": processor.tokenizer.convert_ids_to_tokens(inputs["input_ids"][0].detach().cpu().tolist()),
        "image_grid_thw": inputs.get("image_grid_thw", None).detach().cpu() if "image_grid_thw" in inputs else None,
        "attentions": tuple(a.detach().cpu() for a in outputs.attentions),
    }
    path = OUTPUT_DIR / f"sample_{sample_id}.pt"
    torch.save(bundle, path)
    print(sample_id, answer, "->", path)


In [ ]:
for sample in samples:
    collect_sample(*sample)
